# REST APIs and JSON Responses

## Introduction

So far you have loaded JSON from local files. In practice, most JSON data arrives over the network from an API (Application Programming Interface). This notebook covers how HTTP-based REST APIs work, how to make requests with the `requests` library, and how to handle a JSON response you know the schema of in advance.

## Objectives

You will be able to:

- Describe the request/response cycle and common HTTP methods
- Make a GET request with `requests` and parse the JSON response
- Read an API schema diagram and navigate the response accordingly
- Extract structured data from a known-schema API response into a DataFrame

---

## How REST APIs Work

A **REST API** exposes data at URLs called **endpoints**. Your code (the *client*) sends an HTTP request; the server sends back a response — usually JSON.

| HTTP Method | Meaning | When you use it |
|-------------|---------|------------------|
| `GET` | Retrieve data | Reading data from an API |
| `POST` | Send new data | Creating a record, submitting a form |
| `PUT`/`PATCH` | Update existing data | Editing a record |
| `DELETE` | Remove data | Deleting a record |

As a data scientist you will almost always use **GET**.

### Status codes

Every response includes a status code:

| Code | Meaning |
|------|---------|
| 200 | OK — success |
| 400 | Bad Request — your request had an error |
| 401 | Unauthorized — invalid or missing API key |
| 404 | Not Found — endpoint doesn't exist |
| 429 | Too Many Requests — you've hit the rate limit |
| 500 | Server Error — problem on the API's side |

---

## Making a GET Request

The `requests` library is the standard Python HTTP client. Install with `pip install requests` if needed.

We'll use the **ISS Open Notify API** — a free, no-auth API that reports the current position of the International Space Station and who is aboard.

In [ ]:
import requests
import json
import pandas as pd

# GET request — requires an internet connection, no API key needed
response = requests.get('http://api.open-notify.org/astros.json')

print('Status code:', response.status_code)

In [ ]:
# Parse the JSON response body
payload = response.json()
print(type(payload))
payload

In [ ]:
# Convert the 'people' list to a DataFrame
astronauts = pd.DataFrame(payload['people'])
print(f"{payload['number']} people currently aboard the ISS:")
astronauts

---

## Authentication — API Keys

Most real APIs require an API key to identify you and enforce rate limits. Never hardcode a key in a notebook — read it from an environment variable:

```python
import os

api_key = os.environ.get('MY_API_KEY')

# Pass as a query parameter (most common)
response = requests.get(
    'https://api.example.com/data',
    params={'api-key': api_key, 'q': 'search term'}
)

# Or as a header
response = requests.get(
    'https://api.example.com/data',
    headers={'Authorization': f'Bearer {api_key}'}
)
```

---

## Working with a Known JSON Schema

When an API provides documentation, you know the response structure before you make the request. Here is the schema for the NY Times article search API:

<img src="assets/working_with_known_json_schemas/schema_overview.png" width="400">

<img src="assets/working_with_known_json_schemas/schema_detailed.png" width="500">

The schema tells us: `response` → `docs` (list) → each doc has `headline`, `pub_date`, `web_url`, etc.

In [ ]:
# Load a saved NYT API response (same format as a live response)
with open('data/working_with_known_json_schemas/ny_times_response.json') as f:
    nyt_data = json.load(f)

print(nyt_data.keys())

In [ ]:
# Navigate to the docs list — the actual article records
docs = nyt_data['response']['docs']
print(type(docs), '->', len(docs), 'articles')

In [ ]:
# Iterate to extract headlines
for doc in docs:
    print(doc['headline']['main'])

In [ ]:
# Build a DataFrame from the docs list
nyt_df = pd.DataFrame(docs)
nyt_df[['pub_date', 'word_count', 'headline', 'web_url']].head()

In [ ]:
# The headline column is a nested dict — expand it
headline_keys = nyt_df.headline.iloc[0].keys()

for key in headline_keys:
    nyt_df[f'headline_{key}'] = nyt_df.headline.map(lambda x: x[key])

nyt_df[['headline_main', 'pub_date', 'word_count']].head()

---

## Practice

Work with a saved response from the NY Times movie reviews API. The schema:

<img src="assets/working_with_known_json_schemas_lab/nytimes_movie_schema.png" width="500">

<img src="assets/working_with_known_json_schemas_lab/nytimes_movie_schema_detailed.png" width="500">

In [ ]:
# Load data/working_with_known_json_schemas_lab/ny_times_movies.json


In [ ]:
# Build a DataFrame from the 'results' list (top-level key per the schema)


In [ ]:
# How many unique critics are in this response?


In [ ]:
# The 'link' column contains a nested dict with a 'url' key.
# Create a new column 'review_url' by extracting it.


In [ ]:
# How many total results are in the file?


---

## Summary

In this notebook you learned:

- REST APIs use HTTP GET to return JSON data; status 200 = success, 401 = auth problem, 429 = rate limit
- `requests.get(url).json()` makes a request and parses the response in one step
- Always read API keys from environment variables — never hardcode them
- When you know the schema, navigate directly to the records list; `pd.DataFrame(list_of_dicts)` builds the DataFrame
- Nested dict columns expand cleanly with `df[col].map(lambda x: x[key])`

Next: [04 — LLM APIs](04_llm_apis.ipynb)